# Python drills 6: files, paths and formats

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-drills-06-files-and-formats.ipynb)

Companion to the [Files, paths and formats](../learning/python-06-files-and-formats.ipynb) chapter. The exercises read and write CSV, JSON, JSON Lines and text. They use a temporary folder, which the last cell deletes.

Try each exercise in the code cell below its question. Use the sample inputs to compare your result with the expected output. Expand **Show solution** when you want to review the approach.

## Exercise 1: count the non-blank lines

**Difficulty:** Medium · about 10 min

Write `count_nonblank_lines(path)`. Count the lines that contain something other than spaces. Read the file line by line, so a large file never has to fit in memory.

**Sample input and expected output**

```python
# lines.txt contains "one\n\n  \ntwo\n"
count_nonblank_lines(Path("lines.txt"))  # Expected: 2
# An empty file returns 0.
```

In [ ]:
def count_nonblank_lines(path):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def count_nonblank_lines(path):
    with path.open(encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())
```

</details>

## Exercise 2: read a CSV file with typed values

**Difficulty:** Medium · about 10 min

Write `read_orders(path)`. Read the CSV file with `csv.DictReader`. Return a list of dictionaries where `order_id` is an `int` and `amount` is a `float`. The CSV columns are `order_id` and `amount`.

**Sample input and expected output**

```python
# orders.csv contains:
# order_id,amount
# 1,9.5
# 2,10
read_orders(Path("orders.csv"))
# Expected: [{"order_id": 1, "amount": 9.5}, {"order_id": 2, "amount": 10.0}]
```

In [ ]:
import csv

def read_orders(path):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import csv

def read_orders(path):
    with path.open(newline="", encoding="utf-8") as handle:
        return [
            {"order_id": int(row["order_id"]), "amount": float(row["amount"])}
            for row in csv.DictReader(handle)
        ]
```

</details>

## Exercise 3: write JSON Lines with sorted keys

**Difficulty:** Medium · about 10 min

Write `write_jsonl(path, records)`. Write one JSON object per line, with the keys in sorted order. Return the number of records written.

**Sample input and expected output**

```python
write_jsonl(Path("out.jsonl"), [{"b": 1, "a": 2}, {"c": 3}])
# Expected return value: 2
# Expected file content (with a newline after each object):
# {"a": 2, "b": 1}
# {"c": 3}
```


*Hint: `json.dumps(record, sort_keys=True)` gives one object's text.*

In [ ]:
import json

def write_jsonl(path, records):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import json

def write_jsonl(path, records):
    count = 0
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, sort_keys=True) + "\n")
            count += 1
    return count
```

</details>

## Exercise 4: dump a record that has dates and decimals

**Difficulty:** Medium · about 10 min

Write `dump_record(record)`. Return a JSON string with sorted keys. Convert each `date` to its ISO text, and each `Decimal` to a string, so the exact digits are kept. Other values are left as they are.

**Sample inputs and expected outputs**

| Call | Expected output |
| --- | --- |
| `dump_record({'day': date(2024, 1, 15), 'total': Decimal('12.50'), 'id': 7})` | `'{"day": "2024-01-15", "id": 7, "total": "12.50"}'` |
| `dump_record({})` | `'{}'` |
| `dump_record({'x': None, 'ok': True})` | `'{"ok": true, "x": null}'` |


*Hint: `json.dumps` accepts a `default=` function that is called for values it cannot encode.*

In [ ]:
import json
from datetime import date
from decimal import Decimal

def dump_record(record):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import json
from datetime import date
from decimal import Decimal

def dump_record(record):
    def encode(value):
        if isinstance(value, date):
            return value.isoformat()
        if isinstance(value, Decimal):
            return str(value)
        raise TypeError(f"cannot encode {type(value).__name__}")
    return json.dumps(record, default=encode, sort_keys=True)
```

</details>

## Exercise 5: read the first line without the byte-order mark

**Difficulty:** Medium · about 10 min

Write `first_line_without_bom(path)`. Open the file so that a leading byte-order mark is removed, then return the first line without its line ending.

**Sample input and expected output**

```python
# bom.txt starts with a UTF-8 byte-order mark, then "id,name\n1,ana\n".
first_line_without_bom(Path("bom.txt"))  # Expected: "id,name"
# An empty file returns "".
```


*Hint: the `utf-8-sig` encoding strips the mark.*

In [ ]:
def first_line_without_bom(path):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def first_line_without_bom(path):
    with path.open(encoding="utf-8-sig") as handle:
        return handle.readline().rstrip("\n")
```

</details>

## Exercise 6: write a text file atomically

**Difficulty:** Medium · about 10 min

Write `write_text_atomic(path, text)`. Write the text to a temporary file named `<name>.tmp` in the same folder, then `replace` it onto `path`. Return `path`. No `.tmp` file may remain afterwards. On write or replacement failure, keep the original content, remove the temporary file, and re-raise the error.

**Sample input and expected output**

```python
path = Path("report.txt")
write_text_atomic(path, "ready")  # Expected: Path("report.txt")
path.read_text()                 # Expected: "ready"
Path("report.txt.tmp").exists()   # Expected: False
# If replacement fails, preserve the old report and remove report.txt.tmp.
```

In [ ]:
def write_text_atomic(path, text):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def write_text_atomic(path, text):
    tmp = path.with_name(path.name + ".tmp")
    try:
        tmp.write_text(text, encoding="utf-8")
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)
    return path
```

</details>

## Clean up

Delete the temporary folder. Nothing after this cell uses it.